In [1]:
import json
import torch
import os
from sentence_transformers import util
from torch.nn import functional as F

atomic_fact_data = []
atomic_facts_path = os.path.join(os.getcwd(), "data_for_git", "atomic_facts.jsonl")
with open(atomic_facts_path, "r") as f:
    for line in f:
        atomic_fact_data.append(json.loads(line))
original_generations = []
original_generations_path = os.path.join(os.getcwd(), "data_for_git", "responses.jsonl")
with open(original_generations_path, "r") as f:
    for line in f:
        original_generations.append(json.loads(line))


atomic_facts_dict = {}
for query in atomic_fact_data:
    atomic_facts_dict[query["id"]] = query["results"]["sentences_and_atomic_facts"]
len(atomic_facts_dict)

sorted_generations = sorted(original_generations, key=lambda x: len(x["responses"][0]["logprobs"]), reverse=True)

def gnmt_length_penalty(length: int, alpha: float = 0.6) -> float:
    return ((5 + length) ** alpha) / ((5 + 1) ** alpha)
# multiply all logprob my len(logprobs) for every sentence due to an error in the data
for generation in sorted_generations:
    for response in generation["responses"]:
        for logprob in response["logprobs"]:
            logprob["logprob"] *= len(logprob["logprobs"])

In [2]:
all_query_generations_with_atomic_facts = []
for query_idx in range(len(original_generations)):
    generations = original_generations[query_idx]["responses"]

    generations_with_atomic_facts = []
    individual_atomic_facts_with_logprobs = []
    for generation in generations:
        id = generation["id"]
        atomic_facts = atomic_facts_dict.get(id)
        if atomic_facts == None:
            continue
        for sentence, original in zip(atomic_facts, generation["logprobs"]):
            sentence_with_logprob = {
                "logprob": original["logprob"],
                "atomic_facts": sentence[1]
            }
            for individual_atomic_fact in sentence[1]:
                individual_atomic_facts_with_logprobs.append({
                    "logprob": original["logprob"],
                    "atomic_fact": individual_atomic_fact,
                    "from_sequence": id
                })
            generations_with_atomic_facts.append(sentence)
    all_query_generations_with_atomic_facts.append({
        "generations": generations_with_atomic_facts,
        "atomic_facts": individual_atomic_facts_with_logprobs
    })


In [3]:
import requests
import threading
from server import VectorRequest, EntailmentRequest
from typing import List

def is_entailed(atomic_fact1, atomic_fact2, priority):
    url = "http://localhost:8000/classify"
    request = EntailmentRequest(premise=atomic_fact1, hypothesis=atomic_fact2, priority=priority)
    try:
        response = requests.post(url, json=request.model_dump())
        # return response.json()
        return response.json()["label"] == "ENTAILMENT"
    except Exception as e:
        print(f"Error: {e}")
        return f"Error: {e}"


# cluster the atomic facts
def is_bidirectionally_entailed(atomic_fact1, atomic_fact2, priority):
    one_way = is_entailed(atomic_fact1, atomic_fact2, priority)
    if one_way:
        other_way = is_entailed(atomic_fact2, atomic_fact1, priority)
        return one_way and other_way
    return False

def get_embeddings(texts: List[str]):
    url = "http://localhost:8000/vectors"
    request = VectorRequest(texts=texts)
    response = requests.post(url, json=request.model_dump())
    return response.json()["vectors"]


In [4]:
query_with_most_atomic_facts = max(all_query_generations_with_atomic_facts, key=lambda x: len(x["atomic_facts"]))
print(len(query_with_most_atomic_facts["atomic_facts"]))


419


In [5]:
import tqdm.notebook as tqdm
import concurrent.futures

def update_representatives(clusters):
    for cluster in clusters:
        facts = cluster["atomic_facts"]
        
        # If cluster has only 1 item, it is already the representative
        if len(facts) < 2:
            continue
            
        # 1. Gather all embeddings for this cluster into a Tensor
        # Shape: (N_facts, Hidden_Dim) e.g., (15, 1024)
        cluster_vectors = torch.stack([
            f["embedding"] for f in facts
        ])
        
        centroid = torch.mean(cluster_vectors, dim=0)
        
        centroid = F.normalize(centroid, p=2, dim=0)
        scores = torch.matmul(cluster_vectors, centroid)
        
        best_idx = torch.argmax(scores).item()
        
        cluster["representative"] = facts[best_idx]

    return clusters



total_number_of_atomic_facts = sum([len(query["atomic_facts"]) for query in all_query_generations_with_atomic_facts])

fact_counter = {"value": 0}
fact_lock = threading.Lock()

EMBEDDING_CONCURRENCY = 16
embedding_sem = threading.Semaphore(EMBEDDING_CONCURRENCY)

all_query_clusters = [None]*len(all_query_generations_with_atomic_facts)
def cluster_one_query(query, idx, pbar):
    facts = query["atomic_facts"]
    with embedding_sem:
        embeddings = get_embeddings([fact["atomic_fact"] for fact in facts])
    for fact, embedding in zip(facts, embeddings):
        fact["embedding"] = embedding
    
    comparison_count = 0
    clusters = []
    for i, atomic_fact in enumerate(facts):
        with fact_lock:
            pbar.update(1)
        merged = False        
        if clusters:
            rep_embeddings = torch.stack([
                torch.tensor(c["representative"]["embedding"]) for c in clusters
            ])
        
            scores = util.cos_sim(atomic_fact["embedding"], rep_embeddings)[0]
            sorted_indices = torch.argsort(scores, descending=True).tolist()
        else:
            sorted_indices = clusters
        for index in sorted_indices:
            cluster = clusters[index]
            score = scores[index].item()
                
            # Arbitrary threshold to speed it up and avoid edge cases for deberta
            if score < 0.65:
                break  
            representative = cluster["representative"]
            priority = -1*(len(facts)-i)
            entailment = is_bidirectionally_entailed(atomic_fact["atomic_fact"], representative["atomic_fact"], priority=priority)
            comparison_count += 1

            if entailment:
                cluster["atomic_facts"].append(atomic_fact)
                # Update the representative
                # cluster["representative"] = atomic_fact
                merged = True
                break
        
        if not merged:
            clusters.append({
                "representative": atomic_fact,
                "atomic_facts": [atomic_fact]
            })
    all_query_clusters[idx] = {
        "clusters": clusters,
        "comparisons": comparison_count
    }

# --- MAIN EXECUTION ---
MAX_WORKERS = 256
total_facts = sum([len(q["atomic_facts"]) for q in all_query_generations_with_atomic_facts])

indexed_queries = []
for i, q in enumerate(all_query_generations_with_atomic_facts):
    indexed_queries.append((i, q))
sorted_queries = sorted(indexed_queries, key=lambda x: len(x[1]["atomic_facts"]), reverse=True)

print(f"Starting execution with {MAX_WORKERS} threads.")
print(f"Top 3 largest queries have: {[len(x[1]['atomic_facts']) for x in sorted_queries[:3]]} facts.")
# Use a global pbar
with tqdm.tqdm(total=total_facts, desc="Processing Facts") as pbar:
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        
        # Submit all jobs
        # The executor will only start MAX_WORKERS at a time.
        # As one finishes (and frees up its "NLI slot"), a new one starts (and requests vectors).
        futures = []
        for i, query in enumerate(all_query_generations_with_atomic_facts):
            futures.append(executor.submit(cluster_one_query, query, i, pbar))
        
        for future in concurrent.futures.as_completed(futures):
            try:
                future.result() 
            except Exception as e:
                print(f"Cluster Error: {e}")
                raise e

Processing Facts:   0%|          | 0/124643 [00:00<?, ?it/s]

Starting execution with 256 threads.
Top 3 largest queries have: [419, 372, 354] facts.


Processing Facts:   0%|          | 0/124643 [00:00<?, ?it/s]

In [12]:
path = os.path.join(os.getcwd(),"data_for_git", "all_query_clusters.jsonl")
# save to file
with open(path, "w") as f:
    for query in all_query_clusters:
        # remove the "embedding" field for every atomic fact
        for cluster in query["clusters"]:
            if "embedding" in cluster["representative"]:
                cluster["representative"].pop("embedding")
            for atomic_fact in cluster["atomic_facts"]:
                if "embedding" in atomic_fact:
                    atomic_fact.pop("embedding")
        f.write(json.dumps(query) + "\n")

In [ ]:

print(sum([query["comparisons"] for query in all_query_clusters[:1]]))
for query in all_query_clusters[:1]:
    print(query["comparisons"])



query1 = all_query_clusters[0]
print(len(query1["clusters"]))
for cluster in query1["clusters"]:
    # print(cluster["representative"]["atomic_fact"])
    for atomic_fact in cluster["atomic_facts"]:
        pass
        print(atomic_fact["atomic_fact"])
    print()



In [ ]:
def find_smallest_cosine_similarity(all_query_clusters):
    smallest = float('inf')
    cluster_with_smallest_similarity = None
    for query in all_query_clusters:
        smallest_for_query = float('inf')
        for cluster in query["clusters"]:
            for atomic_fact in cluster["atomic_facts"]:
                similarity = util.cos_sim(atomic_fact["embedding"], cluster["representative"]["embedding"])
                if similarity < smallest_for_query:
                    smallest_for_query = similarity
                    cluster_with_smallest_similarity_for_query = cluster
        print(smallest_for_query)
        if smallest_for_query < 0.5:
            for fact in cluster_with_smallest_similarity_for_query["atomic_facts"]:
                print(fact["atomic_fact"])
            print()
        if smallest_for_query < smallest:
            smallest = smallest_for_query
            cluster_with_smallest_similarity = cluster_with_smallest_similarity_for_query
    return smallest, cluster_with_smallest_similarity

score, cluster = find_smallest_cosine_similarity(all_query_clusters[:1])
for fact in cluster["atomic_facts"]:
    print(fact["atomic_fact"])

str1 = "Richard Burkewood Welbourn served as chairman of the Department of Surgery."
str2 = "The D-day landings took place during World War II."

print(is_entailed(str2, str1))

NameError: name 'all_query_clusters' is not defined

In [ ]:
# Iterate over all queries and build an estimate of the average logprob of 
# a cluster based on how many of the sequences they appear in [1-5]
logprob_per_cluster_type = {}
for query in all_query_clusters:
    if len(query["generations"]) < 5:
        continue
    for cluster in query["clusters"]:
        if len(cluster["atomic_facts"]) not in logprob_per_cluster_type:
            logprob_per_cluster_type[len(cluster["atomic_facts"])] = []
        logprob_per_cluster_type[len(cluster["atomic_facts"])].append(sum([fact["logprob"] for fact in cluster["atomic_facts"]]))
average_logprobs = []
for i in range(5):
    average_logprobs.append(sum(logprob_per_cluster_type[i])/len(logprob_per_cluster_type[i]))
    print(f"Average logprob for {i} atomic facts: {average_logprobs[-1]}")

True
True
True
True
True
True


In [ ]:
import matplotlib.pyplot as plt

type = 1
# plot the density estimate of the type
plt.hist(logprob_per_cluster_type[type], density=True)
plt.show()

In [ ]:
# plot all the density estimates
for i in range(5):
    plt.hist(logprob_per_cluster_type[i], density=True)
plt.show()


In [12]:
query1 = all_query_clusters[0]
for cluster in query1["clusters"]:
    print(cluster["representative"]["atomic_fact"])
    for atomic_fact in cluster["atomic_facts"]:
        print(atomic_fact["atomic_fact"])
    print()


The retrieved documents do not contain any information about Katsunosuke Hori.
The retrieved documents do not contain any information about Katsunosuke Hori.
The retrieved documents do not contain any information about Katsunosuke Hori.
The retrieved documents do not contain any information about Katsunosuke Hori.
The retrieved documents do not contain any information about Katsunosuke Hori.

The documents were provided.
The documents were provided.

The provided documents do not contain any information about Katsunosuke Hori.
The provided documents do not contain any information about Katsunosuke Hori.

I cannot provide a biographical summary of this individual.
I cannot provide a biographical summary of this individual.
I cannot provide a biographical statement about this individual.

The biographical summary is unavailable due to the given context.
The biographical summary is unavailable due to the given context.
I cannot provide a biographical summary based on the given context.

T

In [23]:
str1 = "The retrieved documents do not contain any information about Katsunosuke Hori."
str2 = "The provided documents do not contain any information about Katsunosuke Hori."
# str1 = str2


result = is_bidirectionally_entailed(str1, str2)
print(result)



{'status': 'success', 'label': 'ENTAILMENT', 'probabilities': {'CONTRADICTION': 0.16772204637527466, 'NEUTRAL': 0.13006888329982758, 'ENTAILMENT': 0.7022091150283813}}


In [44]:
for cluster in clusters:
    print(cluster["representative"]["atomic_fact"])
    for atomic_fact in cluster["atomic_facts"]:
        print(atomic_fact["atomic_fact"])
    print()

The provided documents do not contain biographical information about Kang Ji-hwan.
The provided documents do not contain biographical information about Kang Ji-hwan.
The retrieved documents do not contain any information about Kang Ji-hwan.
The provided documents do not contain any biographical information about Kang Ji-hwan.
There is no information about Kang Ji-hwan in the given documents.
There is no information given about a person named Kang Ji-hwan.

The provided documents do not contain any biographical information.
The provided documents do not contain any biographical information.

The context is about Kang Daniel.
The context provided is about Kang Daniel.
The context is about Kang Daniel.
The context is about Kang Daniel.

Kang Daniel is an artist.
Kang Daniel appears to be an artist.
Kang Daniel is an artist.
Kang Daniel has been active in the music industry.

Kang Daniel appears to be an entertainer.
Kang Daniel appears to be an entertainer.

There is no information about 

In [96]:
import math
from collections import defaultdict

def calculate_cluster_probabilities_soft_laplace(clusters, all_atomic_facts_flat, alpha=1.0):
    """
    Implements Soft Laplace Smoothing (Rule of Succession).
    Blends 'Soft Counts' (logprobs) with a 'Uniform Prior' (alpha).
    """
    
    # 1. Calculate Soft Counts (Probability Mass) per Cluster
    # We still use Max-Pooling to get the best logprob per sequence
    
    # Step A: Get best logprob per sequence
    sequence_max_logprobs = defaultdict(lambda: float('-inf'))
    for fact in all_atomic_facts_flat:
        seq_id = fact["from_sequence"]
        lp = fact["logprob"] 
        if lp > sequence_max_logprobs[seq_id]:
            sequence_max_logprobs[seq_id] = lp
            
    # Step B: Calculate Total Mass (Z)
    # This is the "Soft N" (Effective Sample Size)
    total_soft_mass = sum(math.exp(lp) for lp in sequence_max_logprobs.values())
    print(total_soft_mass)
    # Step C: Count Total Clusters (K)
    num_clusters = len(clusters)

    # 2. Calculate P(C) with Smoothing
    for cluster in clusters:
        
        # Get the Soft Count for this cluster
        seq_to_best_logprob = defaultdict(lambda: float('-inf'))
        for fact in cluster["atomic_facts"]:
            seq_id = fact["from_sequence"]
            lp = fact["logprob"]
            if lp > seq_to_best_logprob[seq_id]:
                seq_to_best_logprob[seq_id] = lp
        
        cluster_soft_count = sum(math.exp(lp) for lp in seq_to_best_logprob.values())
        
        # --- THE FORMULA ---
        # (Mass + alpha) / (Total Mass + alpha * K)
        alpha = total_soft_mass/num_clusters
        numerator = cluster_soft_count + alpha
        denominator = total_soft_mass + (alpha * 2)#num_clusters)
        
        cluster["probability"] = numerator / denominator

    # Sort
    clusters.sort(key=lambda x: x["probability"], reverse=True)
    
    return clusters

In [97]:
alpha = 1
rated = calculate_cluster_probabilities_soft_laplace(clusters, individual_atomic_facts_with_logprobs, alpha)
print(2/(7))

1.600537763824919
0.2857142857142857


In [98]:
for cluster in rated:
    print(cluster["probability"])
    print("RERPRESENTATIVE: ", cluster["representative"]["atomic_fact"])
    for atomic_fact in cluster["atomic_facts"]:
        print(atomic_fact["atomic_fact"], math.exp(atomic_fact["logprob"]))
    print()


0.8748584198679513
RERPRESENTATIVE:  The provided documents do not contain biographical information about Kang Ji-hwan.
The provided documents do not contain biographical information about Kang Ji-hwan. 0.4218639859821725
The retrieved documents do not contain any information about Kang Ji-hwan. 0.14838907932271705
The provided documents do not contain any biographical information about Kang Ji-hwan. 0.3953925661809227
There is no information about Kang Ji-hwan in the given documents. 0.30332359175399065
There is no information given about a person named Kang Ji-hwan. 0.2235785706954182

0.41045102493902547
RERPRESENTATIVE:  The provided documents do not contain any information about Kang Ji‑hwan.
The provided documents do not contain any information about Kang Ji‑hwan. 0.3174460661695534
The provided documents do not contain any information about Kang Ji-hwan. 0.3174460661695534

0.29509954890253814
RERPRESENTATIVE:  The provided documents do not contain any biographical information.


In [ ]:
import math

# --- 1. Probability Percolation (Fixing the Split Vote) ---
def percolate_cluster_probabilities(clusters, nli_model):
    """
    Allows probability mass to flow between clusters based on Directional Entailment.
    If Cluster A implies Cluster B, then B inherits A's probability mass.
    """
    # Create a copy of probabilities to avoid modifying while iterating
    # (or just simple addition if we assume DAG structure, but N2 is safer for small N)
    original_probs = [c["probability"] for c in clusters]
    new_probs = original_probs[:] # Shallow copy
    
    # Compare every cluster against every other cluster
    for i, cluster_a in enumerate(clusters):
        for j, cluster_b in enumerate(clusters):
            if i == j: continue
            
            # We check Directional Entailment: Does A imply B?
            # rep_a -> rep_b
            rep_a = cluster_a["representative"]["atomic_fact"]
            rep_b = cluster_b["representative"]["atomic_fact"]
            
            # You need a function that returns True/False for ONE-WAY entailment
            # implies(premise, hypothesis)
            if nli_model.implies(rep_a, rep_b):
                # If A implies B, then B is a 'superset' or 'generalization' of A.
                # B gets A's mass added to it.
                new_probs[j] += original_probs[i]

    # Assign new probabilities back to clusters
    # We cap at 1.0 just in case of slight calibration errors
    for i, cluster in enumerate(clusters):
        cluster["percolated_probability"] = min(new_probs[i], 1.0)
    
    return clusters

In [ ]:
import math

def calculate_sequence_uncertainty_scores(sequences, clusters):
    """
    Assigns an uncertainty score to each sequence based on the 
    Semantic Probability of the facts it contains.
    
    Lower Score = Lower Uncertainty (More Confident/Consensual)
    """
    
    # 1. Create a Lookup Map: Fact Instance -> Cluster Probability
    # Since we can't rely on text matching (stripped), we assume the 'atomic_facts' 
    # inside 'clusters' are the SAME dictionary objects as in 'sequences'.
    # If they are copies, we need a unique ID per fact. Assuming objects here:
    
    fact_to_cluster_prob = {}
    
    for cluster in clusters:
        # We clamp probability to avoid log(0)
        # If a cluster has 0 probability (shouldn't happen with proper smoothing), set epsilon.
        p_c = max(cluster.get("probability", 0), 1e-10)
        
        for fact in cluster["atomic_facts"]:
            # Map the specific fact object (or ID) to the cluster's probability
            # Using Python's object identity `id()` is the safest way if they are the same objects
            fact_to_cluster_prob[id(fact)] = p_c

    # 2. Score Each Sequence
    for seq in sequences:
        log_prob_sum = 0.0
        fact_count = 0
        
        # Iterate over ALL facts in the sequence (including duplicates)
        for fact in seq["atomic_facts"]:
            
            # Retrieve the probability of the cluster this fact belongs to
            # If for some reason a fact wasn't clustered (edge case), assume low prob
            p_c = fact_to_cluster_prob.get(id(fact), 1e-10)
            
            log_prob_sum += math.log(p_c)
            fact_count += 1
            
        # 3. Calculate Uncertainty (Negative Log Likelihood)
        if fact_count > 0:
            # We normalize by the number of facts to prevent long sequences 
            # from naturally having higher uncertainty just because they say more things.
            uncertainty = - (log_prob_sum / fact_count)
        else:
            # Edge case: Sequence produced no atomic facts. 
            # High uncertainty, or 0 depending on your philosophy. 
            # Usually implies the model refused to answer or babbled nonsense.
            uncertainty = 5.0 # Arbitrary high penalty
            
        seq["semantic_uncertainty_score"] = uncertainty

    return sequences